# Day 2. Dot Product / Matrix Multiplication

**오늘의 목표**
- Dot Product를 손으로 계산하고, 그 의미("얼마나 비슷한가")를 설명할 수 있다
- 행렬 곱이 "Dot Product를 여러 번 한 것"이라는 걸 이해한다
- shape 규칙 `(m, n) @ (n, p) → (m, p)`를 보고 결과 shape를 바로 말할 수 있다
- `Q = XWq`, `QKᵀ`가 shape 관점에서 무슨 일을 하는지 안다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import torch
torch.manual_seed(0)   # 랜덤 값을 매번 똑같이 나오게 고정

## 1. Dot Product (내적)

길이가 같은 벡터 두 개를 **같은 위치끼리 곱한 뒤 전부 더한 것**입니다.

```text
a = [1, 2, 3]
b = [4, 5, 6]

a · b = 1×4 + 2×5 + 3×6 = 4 + 10 + 18 = 32
```

결과는 **숫자 하나(scalar)** 입니다.

### 1-1. 직접 계산하기

In [ ]:
a = [1, 2, 3]
b = [4, 5, 6]

result = 0
for x, y in zip(a, b):
    result += x * y

print("반복문으로 계산:", result)

### 1-2. PyTorch로 계산하기

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("a * b        :", a * b, "  # 같은 위치끼리 곱하기 (element-wise)")
print("(a * b).sum():", (a * b).sum())
print("torch.dot    :", torch.dot(a, b))
print("a @ b        :", a @ b)

> 💡 `a * b`와 `a @ b`는 **다릅니다!**
> - `*` : 같은 위치끼리 곱하기만 함 → 벡터 그대로
> - `@` : 곱한 뒤 더하기까지 함 → Dot Product (행렬이면 행렬 곱)
>
> Transformer 코드에서 이 둘을 헷갈리면 shape가 완전히 달라집니다.

## 2. Dot Product의 의미: "얼마나 같은 방향인가"

Dot Product는 두 벡터가 **비슷한 방향을 가리킬수록 커지고**, 반대 방향이면 음수가 됩니다.

이게 바로 Attention의 핵심 아이디어입니다.
> Query와 Key의 Dot Product가 크다 = 둘이 관련이 깊다

In [ ]:
cat    = torch.tensor([0.9, 0.8, 0.1])   # 고양이
kitten = torch.tensor([0.8, 0.9, 0.2])   # 새끼고양이
car    = torch.tensor([-0.7, 0.1, 0.9])  # 자동차

print("고양이 · 새끼고양이 =", round((cat @ kitten).item(), 3))
print("고양이 · 자동차    =", round((cat @ car).item(), 3))

위 벡터들은 설명을 위해 제가 만든 가짜 Embedding입니다.
실제 LLM에서는 학습을 통해 **의미가 비슷한 단어들이 비슷한 방향**을 가리키도록 벡터가 만들어집니다.

## 3. 행렬 곱 = Dot Product를 여러 번

`A @ B`의 결과에서 `(i, j)` 위치의 값은

> **A의 i번째 행** · **B의 j번째 열** (Dot Product)

입니다.

```text
A (2×3)          B (3×2)          C = A @ B (2×2)

[1 2 3]          [ 7  8]          [1·7+2·9+3·11   1·8+2·10+3·12]
[4 5 6]    @     [ 9 10]    =     [4·7+5·9+6·11   4·8+5·10+6·12]
                 [11 12]
```

### 3-1. 반복문으로 직접 구현하기

In [ ]:
def my_matmul(A, B):
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, f"안쪽 차원이 달라요: {n} vs {n2}"

    C = torch.zeros(m, p)
    for i in range(m):          # A의 행
        for j in range(p):      # B의 열
            C[i, j] = A[i, :] @ B[:, j]   # i번째 행 · j번째 열
    return C

A = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
B = torch.tensor([[ 7.0,  8.0],
                  [ 9.0, 10.0],
                  [11.0, 12.0]])

print("직접 구현:\n", my_matmul(A, B))
print("torch.matmul:\n", torch.matmul(A, B))
print("A @ B:\n", A @ B)

## 4. Shape 규칙 (오늘 제일 중요!)

```text
(m, n) @ (n, p)  →  (m, p)
     ↑    ↑
     안쪽 숫자가 같아야 곱할 수 있고, 곱하고 나면 사라진다
```

| 왼쪽 | 오른쪽 | 결과 |
|---|---|---|
| `(2, 3)` | `(3, 2)` | `(2, 2)` |
| `(3, 4)` | `(4, 5)` | `(3, 5)` ← Day 1 체크 질문 4번 정답 |
| `(4, 8)` | `(8, 16)` | `(4, 16)` |
| `(3, 4)` | `(3, 4)` | ❌ 에러 (4 ≠ 3) |

In [ ]:
print((torch.randn(3, 4) @ torch.randn(4, 5)).shape)

안쪽 숫자가 다르면 어떤 에러가 나는지 직접 보세요. 이 에러 메시지는 앞으로 정말 자주 보게 됩니다.

In [ ]:
try:
    torch.randn(3, 4) @ torch.randn(3, 4)
except RuntimeError as e:
    print("RuntimeError:", e)

**에러 해결법**: 한쪽을 뒤집으면(transpose) 됩니다.
`(3, 4) @ (3, 4).T` → `(3, 4) @ (4, 3)` → `(3, 3)`

In [ ]:
X = torch.randn(3, 4)
print("X @ X.T :", (X @ X.T).shape)

## 5. LLM과의 연결

### 5-1. `Q = X @ Wq` — 벡터를 다른 공간으로 옮기기

Token 4개, 각 Token 벡터 길이 8인 문장 `X: (T=4, C=8)`이 있습니다.
여기에 Weight 행렬 `Wq: (8, 16)`을 곱하면?

In [ ]:
T, C, head_size = 4, 8, 16

X  = torch.randn(T, C)            # 문장: Token 4개 × 벡터 길이 8
Wq = torch.randn(C, head_size)    # Query를 만드는 Weight

Q = X @ Wq
print("X :", tuple(X.shape))
print("Wq:", tuple(Wq.shape))
print("Q :", tuple(Q.shape), " ← Token 개수(4)는 그대로, 벡터 길이만 8 → 16")

**중요한 관찰**: Token 개수 `T`는 유지되고 각 Token의 벡터 길이만 바뀝니다.
즉 행렬 곱은 **각 Token 벡터를 하나씩 변환**하는 것과 같습니다. (Day 3 Linear Layer의 정체!)

In [ ]:
# 정말 Token별로 따로 변환한 것과 같은지 확인
Q_one_by_one = torch.stack([X[t] @ Wq for t in range(T)])
print("한 번에 곱한 것 == 하나씩 곱한 것 ?", torch.allclose(Q, Q_one_by_one))

### 5-2. `Q @ Kᵀ` — 모든 Token 쌍의 관련도를 한 번에

Attention Score는 **모든 Token 쌍의 Dot Product**를 담은 표입니다.

In [ ]:
Wk = torch.randn(C, head_size)
K  = X @ Wk                       # (4, 16)

scores = Q @ K.T                  # (4, 16) @ (16, 4) → (4, 4)
print("Q   :", tuple(Q.shape))
print("K.T :", tuple(K.T.shape))
print("QKᵀ :", tuple(scores.shape))
print()
print(scores.round(decimals=1))

`scores[i, j]` = **i번째 Token이 j번째 Token을 얼마나 주목하는가**입니다.
Token 4개 × 4개 = 16개의 Dot Product를 행렬 곱 한 번으로 계산했습니다.

이게 2주차 Attention에서 다룰 `softmax(QKᵀ / √dk)V`의 첫 단계입니다.

### 5-3. Batch가 있으면? `(B, T, C) @ (C, H)`

`@`는 **마지막 두 축**으로 행렬 곱을 하고, 앞쪽 축(batch)은 그대로 둡니다.

In [ ]:
B = 2
X_batch = torch.randn(B, T, C)            # (2, 4, 8)
Q_batch = X_batch @ Wq                    # (2, 4, 8) @ (8, 16)
print("Q_batch:", tuple(Q_batch.shape))

K_batch = X_batch @ Wk
scores_batch = Q_batch @ K_batch.transpose(-2, -1)   # 마지막 두 축만 뒤집기
print("scores :", tuple(scores_batch.shape), " ← 문장마다 (4×4) 표가 하나씩")

> 💡 `K.transpose(-2, -1)`은 "마지막 두 축만 뒤집어라"라는 뜻입니다.
> 커리큘럼 학습 원칙 2번에 나온 `q @ k.transpose(-2, -1)`이 바로 이 코드예요.
> batch 축이 있을 때는 `.T` 대신 이걸 씁니다.

## 6. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** `u = [2, -1, 3]`, `v = [1, 4, 2]`의 Dot Product를 **손으로 계산**해서 적으세요.

In [ ]:
u = torch.tensor([2.0, -1.0, 3.0])
v = torch.tensor([1.0, 4.0, 2.0])

my_answer = None  # TODO: 숫자 하나

assert my_answer == (u @ v).item(), "다시 계산해보세요"
print("Q1 정답!")

**Q2.** 결과 shape를 실행하기 **전에** 예측해서 적으세요. 곱할 수 없으면 `None`이라고 적으세요.

In [ ]:
answers = {
    "(5, 2) @ (2, 7)":       "?",  # TODO
    "(4, 8) @ (8, 4)":       "?",  # TODO
    "(3, 6) @ (3, 6)":       "?",  # TODO  (곱할 수 있을까? 없으면 None)
    "(2, 4, 8) @ (8, 32)":   "?",  # TODO
}

def try_shape(a, b):
    try:
        return tuple((torch.randn(*a) @ torch.randn(*b)).shape)
    except RuntimeError:
        return None

checks = {
    "(5, 2) @ (2, 7)":     try_shape((5, 2), (2, 7)),
    "(4, 8) @ (8, 4)":     try_shape((4, 8), (8, 4)),
    "(3, 6) @ (3, 6)":     try_shape((3, 6), (3, 6)),
    "(2, 4, 8) @ (8, 32)": try_shape((2, 4, 8), (8, 32)),
}
for k in checks:
    mark = "✅" if answers[k] == checks[k] else "❌"
    print(f"{mark} {k:22s} 내 답: {answers[k]}")

**Q3.** `X: (T=5, C=8)`를 `(5, 32)`로 바꾸는 Weight 행렬 `W`를 만드세요.
`W`의 shape는 무엇이어야 할까요?

In [ ]:
X = torch.randn(5, 8)
W = None  # TODO: torch.randn(?, ?)

out = X @ W
assert tuple(out.shape) == (5, 32)
print("Q3 정답! W.shape =", tuple(W.shape))

**Q4.** 아래 `Q`, `K`로 Attention Score `(B, T, T)`를 만드세요.
힌트: 5-3을 참고하세요. `.T`가 아니라 마지막 두 축만 뒤집어야 합니다.

In [ ]:
Q = torch.randn(2, 6, 16)   # (B, T, head_size)
K = torch.randn(2, 6, 16)

scores = None  # TODO

assert tuple(scores.shape) == (2, 6, 6)
assert torch.allclose(scores[0], Q[0] @ K[0].T)
print("Q4 정답!", tuple(scores.shape))

## 7. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. `a * b`와 `a @ b`는 무엇이 다른가?
2. Dot Product 값이 크다는 건 두 벡터가 어떤 관계라는 뜻인가?
3. 행렬 곱 `A @ B`에서 결과의 `(i, j)` 값은 어떻게 계산되는가?
4. `(T, C) @ (C, H)`를 하면 Token 개수는 어떻게 되는가? 이게 왜 중요한가?
5. `Q @ K.T`의 결과 shape가 `(T, T)`인 이유는? 이 표의 각 칸은 무엇을 의미하는가?

**내 답:**

1.
2.
3.
4.
5.